# WavLM Supervised Contrastive Fine-tuning on AbjadKids

Original Colab notebook used to fine-tune WavLM-base-plus with Supervised Contrastive Learning on the AbjadKids Arabic-children speech corpus.

This notebook is part of the HAYATY developmental-screening project. It reproduces the embedding-backbone selection experiment described in Section 5.1.4.1 of the thesis.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q datasets huggingface_hub hf_transfer accelerate transformers torchaudio librosa soundfile

In [ ]:
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_datasets_cache"

In [ ]:
from huggingface_hub import login
login()  # paste your Hugging Face access token when prompted

In [ ]:
from huggingface_hub import snapshot_download

LOCAL_DATASET_DIR = "/content/AbjadKids"

snapshot_path = snapshot_download(
    repo_id="Aziz-snoubra/Abjad-Kids",
    repo_type="dataset",
    local_dir=LOCAL_DATASET_DIR,
    local_dir_use_symlinks=False,
    resume_download=True,
    max_workers=16
)

print("Downloaded to:", snapshot_path)

In [ ]:
DATA_DIR = "/content/AbjadKids"

import os

print(os.path.exists(DATA_DIR))
print(os.listdir(DATA_DIR))

In [ ]:
import os
import pandas as pd
from pathlib import Path

audio_exts = [".wav", ".mp3", ".flac", ".ogg", ".m4a"]

rows = []

for category in ["alphabet", "colors", "numbers"]:
    category_dir = Path(DATA_DIR) / category

    for class_dir in category_dir.iterdir():
        if class_dir.is_dir():
            label_name = class_dir.name

            for file in class_dir.rglob("*"):
                if file.suffix.lower() in audio_exts:
                    rows.append({
                        "path": str(file),
                        "category": category,
                        "label_name": label_name
                    })

df = pd.DataFrame(rows)

print("Total files:", len(df))
print("Total labels:", df["label_name"].nunique())
df.head()

In [ ]:
counts = df["label_name"].value_counts()

print("Number of classes:", counts.shape[0])
print("Min samples per class:", counts.min())
print("Max samples per class:", counts.max())
print("Mean samples per class:", counts.mean())

counts.head(20)

In [ ]:
label_names = sorted(df["label_name"].unique())
label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for name, i in label2id.items()}

df["label"] = df["label_name"].map(label2id)

print("Classes:", len(label_names))
print(label_names[:20])
df.head()

In [ ]:
from pathlib import Path
import re

def extract_speaker(path):
    stem = Path(path).stem
    parts = stem.split("_")

    # filename pattern: Label_speaker_uuid
    if len(parts) >= 3:
        return parts[1].strip().lower()

    if len(parts) >= 2:
        return parts[1].strip().lower()

    return "unknown"

df["speaker"] = df["path"].apply(extract_speaker)

print("Total files:", len(df))
print("Labels:", df["label_name"].nunique())
print("Speakers:", df["speaker"].nunique())

df[["label_name", "speaker", "path"]].head(20)

In [ ]:
from sklearn.model_selection import train_test_split

speakers = df["speaker"].unique()

train_speakers, temp_speakers = train_test_split(
    speakers,
    test_size=0.30,
    random_state=42
)

val_speakers, test_speakers = train_test_split(
    temp_speakers,
    test_size=0.50,
    random_state=42
)

train_df = df[df["speaker"].isin(train_speakers)].reset_index(drop=True)
val_df   = df[df["speaker"].isin(val_speakers)].reset_index(drop=True)
test_df  = df[df["speaker"].isin(test_speakers)].reset_index(drop=True)

print("Train:", len(train_df), "speakers:", train_df["speaker"].nunique(), "labels:", train_df["label_name"].nunique())
print("Val:", len(val_df), "speakers:", val_df["speaker"].nunique(), "labels:", val_df["label_name"].nunique())
print("Test:", len(test_df), "speakers:", test_df["speaker"].nunique(), "labels:", test_df["label_name"].nunique())

print("Speaker overlap train/test:", set(train_df["speaker"]) & set(test_df["speaker"]))

In [ ]:
# verify no speaker is shared between splits
train_s = set(train_df["speaker"])
val_s   = set(val_df["speaker"])
test_s  = set(test_df["speaker"])

print("Train-Val :", train_s & val_s)
print("Train-Test:", train_s & test_s)
print("Val-Test  :", val_s & test_s)

# verify no file is shared between splits
train_p = set(train_df["path"])
val_p   = set(val_df["path"])
test_p  = set(test_df["path"])

print("File overlap Train/Val :", len(train_p & val_p))
print("File overlap Train/Test:", len(train_p & test_p))
print("File overlap Val/Test  :", len(val_p & test_p))

# missing classes in val/test
all_labels = set(df["label_name"])

missing_val = all_labels - set(val_df["label_name"])
missing_test = all_labels - set(test_df["label_name"])

print("Missing labels in Val:", missing_val)
print("Missing labels in Test:", missing_test)

In [ ]:
df_clean = df[df["label_name"] != "Walad"].reset_index(drop=True)

print("Before:", df["label_name"].nunique(), len(df))
print("After:", df_clean["label_name"].nunique(), len(df_clean))

In [ ]:
from sklearn.model_selection import train_test_split

# use the cleaned dataframe
df = df_clean.copy()

# rebuild label index after exclusion
label_names = sorted(df["label_name"].unique())
label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for name, i in label2id.items()}

df["label"] = df["label_name"].map(label2id)

speakers = df["speaker"].unique()

train_speakers, temp_speakers = train_test_split(
    speakers,
    test_size=0.30,
    random_state=42
)

val_speakers, test_speakers = train_test_split(
    temp_speakers,
    test_size=0.50,
    random_state=42
)

train_df = df[df["speaker"].isin(train_speakers)].reset_index(drop=True)
val_df   = df[df["speaker"].isin(val_speakers)].reset_index(drop=True)
test_df  = df[df["speaker"].isin(test_speakers)].reset_index(drop=True)

print("Train:", len(train_df), "files | speakers:", train_df["speaker"].nunique(), "| labels:", train_df["label_name"].nunique())
print("Val:", len(val_df), "files | speakers:", val_df["speaker"].nunique(), "| labels:", val_df["label_name"].nunique())
print("Test:", len(test_df), "files | speakers:", test_df["speaker"].nunique(), "| labels:", test_df["label_name"].nunique())

all_labels = set(df["label_name"])
print("Missing labels in Val:", all_labels - set(val_df["label_name"]))
print("Missing labels in Test:", all_labels - set(test_df["label_name"]))

In [ ]:
!pip install -q transformers torchaudio soundfile librosa tqdm

In [ ]:
import os, random, re, glob
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
import librosa

from torch.utils.data import Dataset, DataLoader, Sampler
from sklearn.model_selection import train_test_split
from transformers import Wav2Vec2FeatureExtractor, WavLMModel
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

In [ ]:
SR = 16000
MAX_SECONDS = 2.5
MAX_LEN = int(SR * MAX_SECONDS)

class AbjadAudioDataset(Dataset):
    def __init__(self, dataframe, sr=16000, max_len=40000):
        self.df = dataframe.reset_index(drop=True)
        self.sr = sr
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):
        wav, sr = sf.read(path, dtype="float32")

        if wav.ndim > 1:
            wav = wav.mean(axis=1)

        if sr != self.sr:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=self.sr)

        wav = wav.astype(np.float32)

        peak = np.max(np.abs(wav)) + 1e-8
        wav = wav / peak

        if len(wav) > self.max_len:
            start = random.randint(0, len(wav) - self.max_len)
            wav = wav[start:start+self.max_len]
        else:
            pad = self.max_len - len(wav)
            wav = np.pad(wav, (0, pad))

        return wav

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav = self.load_audio(row["path"])
        label = int(row["label"])

        return {
            "input_values": torch.tensor(wav, dtype=torch.float32),
            "label": torch.tensor(label, dtype=torch.long),
            "label_name": row["label_name"]
        }

train_dataset = AbjadAudioDataset(train_df)
val_dataset   = AbjadAudioDataset(val_df)
test_dataset  = AbjadAudioDataset(test_df)

print(len(train_dataset), len(val_dataset), len(test_dataset))

In [ ]:
class BalancedBatchSampler(Sampler):
    def __init__(self, dataframe, n_classes=8, n_samples=4):
        self.df = dataframe.reset_index(drop=True)
        self.n_classes = n_classes
        self.n_samples = n_samples

        self.label_to_indices = defaultdict(list)
        for idx, label in enumerate(self.df["label"]):
            self.label_to_indices[int(label)].append(idx)

        self.labels = list(self.label_to_indices.keys())
        self.batch_size = n_classes * n_samples

    def __iter__(self):
        while True:
            selected_labels = random.sample(self.labels, self.n_classes)
            batch = []

            for label in selected_labels:
                indices = self.label_to_indices[label]
                chosen = random.choices(indices, k=self.n_samples)
                batch.extend(chosen)

            random.shuffle(batch)
            yield batch

    def __len__(self):
        return len(self.df) // self.batch_size

def collate_fn(batch):
    input_values = torch.stack([x["input_values"] for x in batch])
    labels = torch.stack([x["label"] for x in batch])
    return {
        "input_values": input_values,
        "labels": labels
    }

train_sampler = BalancedBatchSampler(train_df, n_classes=8, n_samples=4)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

batch = next(iter(train_loader))
print(batch["input_values"].shape)
print(batch["labels"].shape)
print(torch.unique(batch["labels"], return_counts=True))

In [ ]:
MODEL_NAME = "microsoft/wavlm-base-plus"

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)

class WavLMSupConModel(nn.Module):
    def __init__(self, model_name=MODEL_NAME, projection_dim=128):
        super().__init__()

        self.wavlm = WavLMModel.from_pretrained(model_name)
        hidden_size = self.wavlm.config.hidden_size

        self.projection = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, projection_dim)
        )

    def forward(self, input_values):
        outputs = self.wavlm(input_values=input_values)
        x = outputs.last_hidden_state

        x = x.mean(dim=1)

        z = self.projection(x)
        z = F.normalize(z, dim=1)

        return z

model = WavLMSupConModel().to(DEVICE)
print("Model ready")

In [ ]:
# Fine-tune WavLM: train all encoder layers except the first 4 (low-level features kept frozen)
for param in model.wavlm.parameters():
    param.requires_grad = True

for layer in model.wavlm.encoder.layers[:4]:
    for param in layer.parameters():
        param.requires_grad = False

for param in model.projection.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print("Trainable params:", trainable)
print("Total params:", total)
print("Trainable ratio:", round(trainable / total * 100, 2), "%")

In [ ]:
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.05):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]

        labels = labels.view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        logits = torch.matmul(features, features.T) / self.temperature
        logits = logits - logits.max(dim=1, keepdim=True)[0].detach()

        logits_mask = torch.ones_like(mask) - torch.eye(batch_size, device=device)
        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask

        log_prob = logits - torch.log(
            exp_logits.sum(dim=1, keepdim=True) + 1e-12
        )

        positives = mask.sum(dim=1)

        mean_log_prob_pos = (
            (mask * log_prob).sum(dim=1) / (positives + 1e-12)
        )

        loss = -mean_log_prob_pos.mean()
        return loss

criterion = SupConLoss(temperature=0.05)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5,
    weight_decay=1e-4,
    betas=(0.9, 0.98)
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=2000,
    eta_min=1e-6
)

print("Loss + optimizer + scheduler ready")

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/AbjadKids_WavLM_SupCon"
CKPT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
FINAL_DIR = os.path.join(PROJECT_DIR, "final_model")

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

print("CKPT_DIR:", CKPT_DIR)

def save_ckpt(model, optimizer, scheduler, step):
    path = os.path.join(CKPT_DIR, f"wavlm_supcon_step_{step}.pt")
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "step": step,
        "label2id": label2id,
        "id2label": id2label,
    }, path)
    print(f"Saved: {path}")

def load_latest_ckpt(model, optimizer, scheduler):
    files = [f for f in os.listdir(CKPT_DIR) if f.endswith(".pt")]

    if not files:
        print("No checkpoint found. Starting fresh.")
        return 0

    files = sorted(
        files,
        key=lambda x: int(x.split("_step_")[1].replace(".pt", ""))
    )

    latest = files[-1]
    path = os.path.join(CKPT_DIR, latest)

    ckpt = torch.load(path, map_location=DEVICE)

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    print(f"Resumed from: {path}")
    return ckpt["step"]

start_step = 0
print("Starting new optimizer schedule from step 0")

In [ ]:
TOTAL_STEPS = 2000
SAVE_EVERY = 100
PRINT_EVERY = 10

model.train()
loader_iter = iter(train_loader)

pbar = tqdm(
    range(start_step, TOTAL_STEPS),
    initial=start_step,
    total=TOTAL_STEPS
)

for step in pbar:
    batch = next(loader_iter)

    input_values = batch["input_values"].to(DEVICE, non_blocking=True)
    labels = batch["labels"].to(DEVICE, non_blocking=True)

    optimizer.zero_grad()

    embeddings = model(input_values)
    loss = criterion(embeddings, labels)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()
    scheduler.step()

    if step % PRINT_EVERY == 0:
        lr = scheduler.get_last_lr()[0]
        pbar.set_description(
            f"Step {step} | Loss {loss.item():.4f} | LR {lr:.2e}"
        )

    if step > 0 and step % SAVE_EVERY == 0:
        save_ckpt(model, optimizer, scheduler, step)

save_ckpt(model, optimizer, scheduler, TOTAL_STEPS)
print("Training finished")

In [ ]:
model.eval()

test_eval_df = test_df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(min(len(x), 20), random_state=42)
).reset_index(drop=True)

test_eval_dataset = AbjadAudioDataset(test_eval_df)

test_eval_loader = DataLoader(
    test_eval_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

all_embs = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_eval_loader):
        input_values = batch["input_values"].to(DEVICE)
        labels = batch["labels"].cpu().numpy()

        embs = model(input_values).cpu().numpy()

        all_embs.append(embs)
        all_labels.extend(labels)

all_embs = np.vstack(all_embs)
all_labels = np.array(all_labels)

print("Embeddings:", all_embs.shape)
print("Labels:", all_labels.shape)
print("Classes:", len(set(all_labels)))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sim = cosine_similarity(all_embs)

# do not compare a sample with itself
np.fill_diagonal(sim, -999)

top1_correct = 0
top5_correct = 0

for i in range(len(all_labels)):
    ranking = np.argsort(sim[i])[::-1]

    top1 = ranking[0]
    top5 = ranking[:5]

    if all_labels[top1] == all_labels[i]:
        top1_correct += 1

    if all_labels[i] in all_labels[top5]:
        top5_correct += 1

top1_acc = top1_correct / len(all_labels)
top5_acc = top5_correct / len(all_labels)

print("Top-1 Retrieval Accuracy:", round(top1_acc * 100, 2), "%")
print("Top-5 Retrieval Accuracy:", round(top5_acc * 100, 2), "%")

In [ ]:
same_sims = []
diff_sims = []

for i in range(len(all_labels)):
    for j in range(i + 1, len(all_labels)):
        if all_labels[i] == all_labels[j]:
            same_sims.append(sim[i, j])
        else:
            diff_sims.append(sim[i, j])

same_sims = np.array(same_sims)
diff_sims = np.array(diff_sims)

print("Same-word similarity mean:", round(same_sims.mean(), 4))
print("Different-word similarity mean:", round(diff_sims.mean(), 4))
print("Gap:", round(same_sims.mean() - diff_sims.mean(), 4))

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix
import numpy as np

def purity_score(y_true, y_pred):
    matrix = contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(matrix, axis=0)) / np.sum(matrix)

# cosine distance
clustering = AgglomerativeClustering(
    n_clusters=len(set(all_labels)),
    metric="cosine",
    linkage="average"
)

pred_clusters = clustering.fit_predict(all_embs)

ari = adjusted_rand_score(all_labels, pred_clusters)
nmi = normalized_mutual_info_score(all_labels, pred_clusters)
purity = purity_score(all_labels, pred_clusters)

print("ARI:", round(ari, 4))
print("NMI:", round(nmi, 4))
print("Purity:", round(purity, 4))

In [ ]:
import hdbscan
import numpy as np
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix

def purity_score(y_true, y_pred):
    matrix = contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(matrix, axis=0)) / np.sum(matrix)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=2,
    metric='euclidean'
)

pred_clusters = clusterer.fit_predict(all_embs)

# remove noise points (label == -1) from the evaluation
mask = pred_clusters != -1

y_true = all_labels[mask]
y_pred = pred_clusters[mask]

ari = adjusted_rand_score(y_true, y_pred)
nmi = normalized_mutual_info_score(y_true, y_pred)
purity = purity_score(y_true, y_pred)

print("Clusters found:", len(set(y_pred)))
print("Noise removed:", np.sum(pred_clusters == -1))
print("ARI:", round(ari, 4))
print("NMI:", round(nmi, 4))
print("Purity:", round(purity, 4))